# Equatorial waves and tidally locked exoplanets

## Recap 

### Equatorial waves
Near the equator of a planet, $f\rightarrow 0$, $\text{Ro}\rightarrow\infty$, so locally we can say $f \approx \beta y$. This is known as the **equatorial beta plane** approximation

Under this approximation, several types of waves can be excited. These include:
* Inertia gravity waves (see exercise 2)
* Equatorial Kelvin waves (travelling eastwards with speed $c = 1$)
* Equatorial Rossby waves (similar, but not identical to those in exercise 3)

<img src="./images/mg.png" alt="drawing" width="70%"/>

### Forcing in the shallow water model

To simulate a planet that is being forced with stellar radiation, we will add a "height source". In this section, it will be useful to think of the height of a layer as proxy for the temperature. We will also include a Rayleigh-style drag to the momentum equations.

$$
\begin{align}
\partial_t h + \partial_x(hu) + \partial_y(hv) &= -\frac{h - h_{eq}}{\tau_h} \\
\partial_t u + u\partial_x u + v \partial_y v - \beta y v + \partial_x h &= -\frac{u}{\tau_u}\\
\partial_t v + u\partial_x v + v\partial_y v + \beta y u + \partial_y h &= -\frac{v}{\tau_v}
\end{align}
$$

where $\tau$ values are drag timescales. The value of $h_{eq}$ is the "equilibrium" $h$ value that the forcing is driving our model towards. In our shallow water model, we can control the rayleigh drag timescales from the `cfg` file, and the forcing in the `forcing.py` file. 



### For those using colab + google drive, run cells below

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

! pip install -e /content/drive/MyDrive/SW_summerschool/

In [ ]:
import sys
import importlib
sys.modules["imp"] = importlib
from google.colab import output
output.enable_custom_widget_manager()

### Now restart kernel using Runtime/Restart session in the colab menu

The config/output files for this exercise should be found/put in the directory `/content/drive/MyDrive/SW_summerschool/examples/Exoplanets`

In [ ]:
# Need to change into directory with config files
%cd /content/drive/MyDrive/SW_summerschool/examples/Exoplanets

### For those with local install, continue from below

In [ ]:
# Load the model again
%load_ext autoreload
%autoreload 2
%matplotlib widget
from IPython.display import HTML
import xarray as xr
import numpy as np
from sw_summerschool.plotting import animate_height_velocity, animate_contour
from sw_summerschool import SW_model
from sw_summerschool.helper import interp_to_centre
import matplotlib as mpl
import matplotlib.pyplot as plt


### Forcing function

To mimic the forcing applied by the star on a tidally-locked exoplanet, we will use the forcing function:
$$
h_{eq} =1 + A\cos(\pi x)\cos(\pi y)
$$
with the amplitude $A < 1$ to avoid negative heights. 

**Task**

Implement this forcing function in `forcing.py` 


In [ ]:
# Model must be run longer to make up for the longer wave speed
model = SW_model("cfg_exoplanet.yaml", outfile = 'exoplanet.nc', tstep = 1.e-3, io_freq = 1000, print_freq = 1000, Ro=10000, beta =1.0)

In [ ]:
model.integrate(30000)

## Tasks

1. Run the model with this forcing, trying different model parameters. Vary the following.
    1. **The drag timescales** (`tau_h` and `tau_u`). In our non-dimensionalised coordinates, any $\tau\ll 1$ is a case with "strong drag" (i.e. the timescale over which speeds are damped and the height field is forced towards the equilibrium height field is $<1$). $\tau\gg 1$ is "weak drag", i.e. the timescale over which drag acts is long with respect to other timescales. Aside: in our non-dimensionalised coordinates, gravity waves travel at $c=1$, so a timespan $\delta t = 1$ corresponds to roughly the length of time a gravity wave takes to cross the domain.
  
    2. **The beta parameter, $\beta$**. Try $\beta = 0$, $\beta = 1$ and $\beta = 5$. How does the height field and flow change?
  
2. In the steady state, it can be shown that the energy balance is given by:
$$
\nabla\cdot(\mathbf{u} h F) = QF - \frac{h\mathbf{u}^2}{\tau_u}
$$
where $F = \mathbf{u}^2/2 + h$, and $Q$ is the forcing. The left hand side represents the rate at which energy flows out of a region. When the divergence of the term in brackets is positive, energy flows out of this region. When the divergence of the term in brackets is negative, energy flows into the region.  Plot the quantity on the left hand side $\nabla\cdot(\mathbf{u} h F)$ and show that the flow acts to redistribute the energy inputted by forcing. 



### Extension tasks

In a shallow water model of giant exoplanets, we tend to think of the equations as describing an active, dynamic weather layer overlying a deep, quiescent interior. Heating acts to transfer mass from this deeper layer up into the dynamic layer. Since this lower, quiescent layer has $\mathbf{u} = 0$, we need to adjust the momentum equation (otherwise mass injected into the upper layer is instantly accelerated to a speed $\mathbf{u}$, which adds a spurious source/sink of momentum). To adjust for this, we should alter the momentum equation to:

$$
\frac{d \mathbf{u}}{dt} + \text{Ro}^{-1}\mathbf{f}\times\mathbf{u} + \nabla h= \begin{cases}
0, & Q < 0,\\
-Q\mathbf{u}/h, & Q>0.
\end{cases}
$$

Tasks:
* In the regular case, where this momentum term is neglected, plot a the mean zonal (east-west) wind $\bar{u}$ as a function of $y$, in the steady state. Is there positive $u$ ("superrotation") at the equator?
* Now include the momentum term above, and plot the same values. What do you notice? 

### Appendix: Derivation of Kelvin waves
The Kelvin wave mode appears at the equator in the equatorial beta plane approximation. Assume that $v = 0$ and that the $h$ and $u$ variables can be expanded as plane waves in the $x$ and $t$ dimensions:
$$
(h, u) = (\tilde{h}(y), \tilde{u}(y))\exp(i(\omega t - k x))
$$
From the shallow water equations, we get:
$$
\begin{align}
\omega\tilde{h} - k\tilde{u} &= 0\\
\omega \tilde{u} - k\tilde{h} &= 0 \\
\beta y \tilde{u} + \frac{d\tilde{h}}{dy} &= 0
\end{align}
$$

The first two equations combine to give $\omega = \pm k$ and $\tilde{u} = \pm \tilde{h}$, i.e. the wave can travel either in the positive $x$ or negative $x$ direction. Eliminating $\tilde{u}$ in the third equation gives:

$$
\frac{d \tilde{h}}{dy} = \mp \beta y \tilde{h} 
$$
which can be solved for $\tilde{h}$:
$$
\tilde{h} = \tilde{h}_0 \exp\left(\mp\frac{\beta y^2}{2}\right)
$$
We want solutions that decrease away from the equator (as $y\rightarrow\infty$), meaning we need to select the $\omega = +k$ solution (the eastward travelling solution), and discard the $\omega = -k$ solution (the westward travelling solution).